# Graph-Grounded Reddit Emotion Intelligence RAG

A reproducible notebook for political Reddit analysis: classify all GoEmotions labels, build a conversation graph, and retrieve source evidence with LangChain.

The benchmark uses one majority label per comment. Ties are removed, and the full 28-label task is evaluated on a stratified held-out set.

## Business problem

An analyst studying political Reddit discussions needs more than a sentiment score. They need to find emotional reactions, compare subreddits, follow replies, and inspect the comments behind a result.

In [ ]:
from pathlib import Path
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

DATA = Path('../data/goemotions')
OUT = Path('../outputs/notebook_showcase')
OUT.mkdir(parents=True, exist_ok=True)
raw = pd.concat([pd.read_csv(p) for p in glob.glob(str(DATA / 'goemotions_*.csv'))], ignore_index=True)
raw.shape

In [ ]:
# Collapse rater rows to one high-confidence emotion label per comment.
emotion_cols = [c for c in raw.columns if c not in {'text','id','author','subreddit','link_id','parent_id','created_utc','rater_id','example_very_unclear'}]
votes = raw.groupby('id')[emotion_cols].sum()
texts = raw.drop_duplicates('id').set_index('id')
rows = []
for comment_id, row in votes.iterrows():
    labels = [e for e, value in row.items() if value == row.max() and value > 0]
    if len(labels) == 1:
        record = texts.loc[comment_id]
        rows.append({'text': record.text, 'label': labels[0], 'id': comment_id, 'author': record.author, 'subreddit': record.subreddit, 'parent_id': record.parent_id, 'created_utc': record.created_utc})
data = pd.DataFrame(rows)
data.label.value_counts().sort_index()

In [ ]:
# Leakage-safe 28-class benchmark. The test set is never used during fitting.
train, test = train_test_split(data, test_size=.2, random_state=42, stratify=data.label)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True, max_features=50000)
X_train = vectorizer.fit_transform(train.text)
X_test = vectorizer.transform(test.text)
model = LinearSVC(C=1)
model.fit(X_train, train.label)
pred = model.predict(X_test)
metrics = {'accuracy': accuracy_score(test.label, pred), 'macro_f1': f1_score(test.label, pred, average='macro'), 'weighted_f1': f1_score(test.label, pred, average='weighted'), 'classes': data.label.nunique(), 'rows': len(data), 'test_rows': len(test)}
metrics

In [ ]:
print(classification_report(test.label, pred))
labels = sorted(data.label.unique())
ConfusionMatrixDisplay.from_predictions(test.label, pred, labels=labels, display_labels=labels, xticks_rotation=90, cmap='Blues', colorbar=False)
plt.title('Held-out 28-class GoEmotions confusion matrix')
plt.tight_layout()
plt.savefig(OUT / 'full_goemotions_confusion_matrix.png', dpi=180)
plt.show()

In [ ]:
# Conversation graph: comments, subreddits, and parent-child edges.
graph = nx.DiGraph()
for row in data.itertuples():
    graph.add_node(str(row.id), node_type='comment', emotion=row.label, subreddit=row.subreddit)
    graph.add_node(f'subreddit:{row.subreddit}', node_type='subreddit')
    graph.add_edge(str(row.id), f'subreddit:{row.subreddit}', edge_type='posted_in')
for row in data.dropna(subset=['parent_id']).itertuples():
    graph.add_edge(str(row.parent_id), str(row.id), edge_type='reply')
print({'nodes': graph.number_of_nodes(), 'edges': graph.number_of_edges(), 'weak_components': nx.number_weakly_connected_components(graph)})
community = data.groupby('subreddit').agg(comments=('id','count'), positive_rate=('label', lambda x: (x.isin(['admiration','amusement','approval','caring','desire','excitement','gratitude','joy','love','optimism','pride','relief'])).mean())).sort_values('comments', ascending=False).head(15)
community.plot.bar(y='positive_rate', legend=False, figsize=(10,4), title='Positive emotion rate by largest communities')
plt.ylabel('positive emotion rate'); plt.tight_layout(); plt.savefig(OUT / 'community_polarity.png', dpi=160); plt.show()

## LangChain RAG

The RAG code in `src/reddit_rag/rag.py` uses a LangChain `BaseRetriever`, BM25 + SPLADE reciprocal-rank fusion, Qdrant sparse vectors, and evidence-cited extractive answers. Run it from the repository root with:

```powershell
$env:PYTHONPATH = (Join-Path (Get-Location) 'src')
.\.venv_sys\Scripts\python.exe -m reddit_rag.rag_cli --politosphere-comments .\data\politosphere\comments_2008-01.bz2 --query "What concerns appear in the discussion?" --limit 100 --top-k 5
```

The answer is intentionally grounded in retrieved evidence and should be reviewed by an analyst; it is not a population-level opinion estimate.